# Dealing with Mysterious Conflicts

This notebook shows *reduce/reduce* conflicts that result from the fact that the set of `LR` states is
compressed into the set of `LALR` states.  Such a conflict is called *mysterious*, because it is not caused
by any defect of the grammar:  the grammar below is unambiguous and it even has the `LR(1)` property.  The
conflict appears only when two `LR` states that have the same *core* are merged into a single `LALR` state.

We discuss the following grammar:
```
    s : "v" a "y"
      | "w" b "y"
      | "v" b "z"
      | "w" a "z"

    a : X

    b : X

    X : "x"
```
The language generated by this grammar consists of the four strings `vxy`, `vxz`, `wxy` and `wxz`.  Whether
the `x` in the middle is an `a` or a `b` is determined by the *combination* of the first and the last
character:  in `vxy` and `wxz` it is an `a`, while in `wxy` and `vxz` it is a `b`.

## Specification of the Grammar

The token classes and the grammar rules are given in a single grammar.  The characters `v`, `w`, `y` and `z`
occur as *anonymous terminals*, that is, as string literals inside the rules.  Names are invented for these
terminals:  for a literal consisting of a single letter, the name is that letter in upper case.  Hence, the
terminals are called `V`, `W`, `Y` and `Z` and these invented names show up in the error message below.

In [ ]:
grammar = r"""
    s : "v" a "y"
      | "w" b "y"
      | "v" b "z"
      | "w" a "z"

    a : X

    b : X

    X : "x"

    %import common.WS
    %ignore WS
"""

## Trying to Build an `LALR` Parser

The messages about conflicts are written to the logger object `lark.logger` and the level of this logger is
set to `logging.CRITICAL` when `Lark` is imported.  In order to see them, we have to lower this level and,
in addition, create the parser with `debug=True`.  Only then is the offending state included in the message.

In [ ]:
import logging

from lark import Lark, GrammarError, logger

logger.setLevel(logging.DEBUG)

In [ ]:
try:
    Lark(grammar, start='s', parser='lalr', debug=True)
    print('No conflict.')
except GrammarError as e:
    print(e)

## Where Does the Conflict Come From?

If we compute the `LR(1)` states of this grammar, we find the two states
$$ \{\; a \rightarrow X \bullet : \texttt{'y'}, \quad b \rightarrow X \bullet : \texttt{'z'} \;\}
   \qquad\text{and}\qquad
   \{\; b \rightarrow X \bullet : \texttt{'y'}, \quad a \rightarrow X \bullet : \texttt{'z'} \;\} $$
The first of these states is reached after reading `vx`, the second one after reading `wx`.

Taken by itself, *neither* of these two states has a conflict, since the follow sets of the two rules are
disjoint in both cases.  This is precisely the reason why the grammar has the `LR(1)` property.

However, the two states have the same *core*
$$ \{\; a \rightarrow X \bullet, \quad b \rightarrow X \bullet \;\} $$
and therefore they are merged when the `LALR` table is computed.  The resulting state is
$$ \{\; a \rightarrow X \bullet : \{\texttt{'y'}, \texttt{'z'}\}, \quad
       b \rightarrow X \bullet : \{\texttt{'y'}, \texttt{'z'}\} \;\} $$
and this state obviously has a *reduce/reduce* conflict, both for the token `'y'` and for the token `'z'`.
This is exactly what the error message above reports.  Unfortunately, `Lark` does not support the follow sets.

The merger has thrown away the information about which of the two states we came from, and it is precisely
this information that would have been needed to decide between `a` and `b`.  The grammar is therefore an
`LR(1)` grammar, but *not* an `LALR(1)` grammar.

Note that there is no parse table to inspect:  since the exception is raised *during* the construction of
the table, no parser object is created.

## A Way Out: The `Earley` Parser

The *default* parser of `Lark` is an `Earley` parser and the `Earley` algorithm does not build a parse table
at all.  Therefore, the grammar is accepted as it stands and all four strings of the language are parsed
correctly.

The keyword argument `keep_all_tokens=True` tells `Lark` to keep the anonymous terminals in the parse tree.
Without it, the tokens would be filtered out and the trees below would be nearly empty.

The price for this convenience is a parser that is considerably slower than an `LALR` parser.

In [ ]:
earley_parser = Lark(grammar, start='s', parser='earley', keep_all_tokens=True)

In [ ]:
for s in ['vxy', 'vxz', 'wxy', 'wxz']:
    print(earley_parser.parse(s).pretty())

Observe that the middle `x` is recognized as an `a` in `vxy` and `wxz`, but as a `b` in `vxz` and `wxy`.